# Ablation Study: Impact of `fare_low` on XGBoost Airfare Model

This notebook isolates the contribution of `fare_low` (the market price floor — the average fare paid by the most budget-conscious passengers on a route) by training two otherwise-identical XGBoost models:

| Model | Features |
|---|---|
| **Full Model** | All 11 features including `fare_low` |
| **Ablation Model** | Same 11 features, minus `fare_low` |

The Δ in R², MAE, and RMSE quantifies how much predictive signal `fare_low` adds beyond the remaining structural features (distance, demand, market share, and hub characteristics).

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load Dataset

In [ ]:
file_path = 'airline_ticket_dataset.xlsx'
df = pd.read_excel(file_path)
print(f"Dataset shape: {df.shape}")
df.head(3)

## 2. Define Features & Target

In [ ]:
features = [
    # Layer 1 — Baseline
    'nsmiles',
    'passengers',
    # Layer 2 — Competition
    'large_ms',
    'lf_ms',
    'fare_low',
    # Layer 3 — Hub characteristics (both endpoints)
    'TotalFaredPax_city1',
    'TotalPerLFMkts_city1',
    'TotalPerPrem_city1',
    'TotalFaredPax_city2',
    'TotalPerLFMkts_city2',
    'TotalPerPrem_city2',
]

X = df[features].fillna(0)
y = df['fare']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")

## 3. Train Full Model (all 11 features)

In [ ]:
print("Training full model...")
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n── Full Model Performance ──")
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : ${mae:.2f}")
print(f"  RMSE : ${rmse:.2f}")

## 4. Ablation Study: Model Without `fare_low`

`fare_low` represents the average fare paid by the most budget-conscious passengers on a route — effectively a market price floor set by competition. Removing it tests whether the remaining structural features (distance, demand, market concentration, hub scale) alone can explain airfare variation.

In [ ]:
# ── Ablation Study: Model Without fare_low ──
features_no_farelow = [f for f in features if f != 'fare_low']

X_abl = df[features_no_farelow].fillna(0)
X_abl_train, X_abl_test, y_abl_train, y_abl_test = train_test_split(
    X_abl, y, test_size=0.2, random_state=42
)

print("Training ablation model (no fare_low)...")
model_abl = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model_abl.fit(X_abl_train, y_abl_train)

y_abl_pred = model_abl.predict(X_abl_test)
r2_abl   = r2_score(y_abl_test, y_abl_pred)
mae_abl  = mean_absolute_error(y_abl_test, y_abl_pred)
rmse_abl = np.sqrt(mean_squared_error(y_abl_test, y_abl_pred))

print("\n── Ablation Study: Model Comparison ──")
print(f"{'Metric':<8}  {'Full Model':>12}  {'No fare_low':>12}  {'Δ':>10}")
print(f"{'R²':<8}  {r2:>12.4f}  {r2_abl:>12.4f}  {r2_abl - r2:>+10.4f}")
print(f"{'MAE':<8}  ${mae:>11.2f}  ${mae_abl:>11.2f}  ${mae_abl - mae:>+9.2f}")
print(f"{'RMSE':<8}  ${rmse:>11.2f}  ${rmse_abl:>11.2f}  ${rmse_abl - rmse:>+9.2f}")

## 5. SHAP Feature Importance — Ablation Model

With `fare_low` removed, how does the model redistribute importance among the remaining features?

In [ ]:
sample_size_abl   = int(len(X_abl_test) * 0.8)
X_abl_test_sample = X_abl_test.sample(n=sample_size_abl, random_state=42)

print(f"Calculating SHAP values for ablation model ({sample_size_abl} samples)...")
explainer_abl   = shap.TreeExplainer(model_abl)
shap_values_abl = explainer_abl.shap_values(X_abl_test_sample)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_abl, X_abl_test_sample, plot_type="bar", show=False)
plt.title(
    "SHAP: Feature Importance — Model Without fare_low\n"
    f"R²={r2_abl:.4f}  MAE=${mae_abl:.2f}  RMSE=${rmse_abl:.2f}  "
    f"(vs. full model R²={r2:.4f})",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("shap_bar_no_farelow.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → shap_bar_no_farelow.png")

## 6. Interpretation

| Metric | Full Model | No `fare_low` | Change |
|--------|-----------|----------------|--------|
| R² | — | — | Δ |
| MAE | — | — | Δ |
| RMSE | — | — | Δ |

*(Fill in after running cells above.)*

**Key takeaways:**
- If R² drops significantly, `fare_low` encodes route-level competitive information that structural features cannot replicate.
- If R² is mostly preserved, the remaining features (distance, demand, market share, hub scale) explain the bulk of airfare variation independently.
- The SHAP bar plot above shows how importance shifts to `large_ms` and `nsmiles` when the price-floor signal is removed.